# Brain Explorer step 1-4 walkthrough

Manual check for the offline reference pipeline + regional BAG/contribution matrices
+ static NiiVue demo built per `plans/app_kickoff.md` (build order 1-4).

Covers:
1. Region centroids (`neuroalign.reference.compute_centroids`)
2. Literature enrichment + `region_reference.json` (build-time contract vs BAG names)
3. Regional BAG / contribution matrices from the multivariate stacker
4. Static NiiVue demo data (`app/viewer/index.html`)

Each section is independently re-runnable; nothing here mutates the committed artifacts
(re-running Step 3's fit writes to `anat_baseline_demo`, not the production `anat_baseline` result).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
%matplotlib inline

## Step 1 — Region centroids

`neuroalign.reference.compute_centroids` loads the atlas `dseg.nii.gz`, computes a
per-label center-of-mass, and converts voxel -> MNI mm via the image affine. Output is
keyed by region **name** (matches the BAG output's `region` column, not `schaefer_017`-style
label IDs - see plan Context).

In [ ]:
centroids = json.loads((ROOT / "data/reference/region_centroids.json").read_text())
print(f"{len(centroids)} regions")
for name in ["LH_Vis_1", "NAc-shell-lh", "pCAU-rh"]:
    print(name, centroids[name])

Sanity: MNI coordinates should sit inside a generous brain-extent bounding box, and
left-hemisphere ("LH"/"lh") regions should have negative x.

In [ ]:
xs, ys, zs = zip(*(c["centroid_mni"] for c in centroids.values()))
print("x range:", min(xs), max(xs))
print("y range:", min(ys), max(ys))
print("z range:", min(zs), max(zs))

lh_x = [c["centroid_mni"][0] for name, c in centroids.items() if name.upper().startswith("LH")]
rh_x = [c["centroid_mni"][0] for name, c in centroids.items() if name.upper().startswith("RH")]
print("LH mean x (expect < 0):", np.mean(lh_x))
print("RH mean x (expect > 0):", np.mean(rh_x))

Glass-brain spot-check image, written by `neuroalign.reference.spot_check` when the
`centroids` CLI step last ran (`python -m neuroalign.reference centroids`).

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=str(ROOT / "data/reference/centroid_spotcheck.png"))

## Step 2 — Literature enrichment + `region_reference.json`

`neuroalign.reference.enrich_literature` decodes Neurosynth terms/studies per region centroid
via NiMARE; `merge_reference` combines centroids + literature + a derived plain-language name
into `app/assets/region_reference.json`, and **fails loud** if the key set doesn't exactly
match the BAG output's region names (the app_kickoff.md build-time contract).

In [ ]:
reference = json.loads((ROOT / "app/assets/region_reference.json").read_text())
print(f"{len(reference)} regions in region_reference.json")

for name in ["LH_Vis_1", "NAc-shell-lh"]:
    info = reference[name]
    print(f"\n{name} -> {info['plain_name']}")
    print("  network:", info["network"], "| structure:", info["structure"])
    print("  centroid_mni:", info["centroid_mni"])
    print("  top terms:", [t["term"] for t in info["terms"][:5]])
    print("  n studies:", len(info["studies"]))

Re-verify the contract directly against the current BAG output (`region_metrics.parquet`)
rather than trusting the file was built correctly - this is the same check
`neuroalign.reference.merge_reference` runs at build time.

In [ ]:
from neuroalign.reference import bag_region_names

expected = bag_region_names()
actual = set(reference.keys())
print("region_reference.json keys == BAG region names:", actual == expected)
print("missing:", sorted(expected - actual))
print("extra:", sorted(actual - expected))

In [ ]:
n_zero_terms = sum(1 for v in reference.values() if not v["terms"])
term_counts = [len(v["terms"]) for v in reference.values()]
print("regions with zero literature terms:", n_zero_terms)
print("median terms/region:", int(np.median(term_counts)))

## Step 3 — Regional BAG / contribution matrices

Per-region signal for coloring the brain: **regional BAG** (stage-1 per-region predicted age
minus chronological age) and the linear meta-learner's **per-region contribution** to the
overall predicted age, both persisted by `MultivariateRegionalBAGEstimator` from a single
full-data fit (no `shap` dependency, no refit) - see plan Context point 3.

This demo run used feature set `["anat_volume_mm3", "anat_thickness_mean_mm"]` and was saved to
`anat_baseline_demo` (a new dir, separate from the committed `anat_baseline` result - the exact
original invocation params for `anat_baseline` were ambiguous, so nothing there was touched).

In [ ]:
bag_dir = ROOT / "data/processed_full/bag/multivariate/anat_baseline_demo"

bag = pd.read_parquet(bag_dir / "bag.parquet")
regional_bag = pd.read_parquet(bag_dir / "regional_bag.parquet")
regional_contribution = pd.read_parquet(bag_dir / "regional_contribution.parquet")
region_metrics = pd.read_parquet(bag_dir / "region_metrics.parquet")

region_cols = [c for c in regional_bag.columns if c not in ("uid", "session_id")]
print("scalar bag.parquet:", bag.shape)
print("regional_bag.parquet:", regional_bag.shape, "-", len(region_cols), "region columns")
print("regional_contribution.parquet:", regional_contribution.shape)
print("region_cols == BAG region_metrics names:", set(region_cols) == set(region_metrics["region"]))

Contribution-sum identity: each session's per-region contributions should sum to
`prediction - intercept_` of the (linear) meta-learner. `tests/test_regional_bag_contribution.py`
checks this on synthetic data; here we sanity-check the saved real-data output distribution
(the exact identity needs the fitted `RidgeCV` intercept, which isn't persisted standalone -
see the pytest test for a from-scratch numeric check).

In [ ]:
row_abs_sum = regional_bag[region_cols].abs().sum(axis=1)
demo_row_idx = row_abs_sum.idxmax()
demo_row = regional_bag.loc[demo_row_idx]
print("Most extreme demo participant:", demo_row["uid"], demo_row["session_id"])

vals = demo_row[region_cols].astype(float)
print("min / median / max regional BAG:", vals.min(), vals.median(), vals.max())
print("n saturated beyond +/-2.5 (viewer colormap clamp):", (vals.abs() > 2.5).sum(), "/", len(vals))

In [ ]:
ax = vals.hist(bins=40, figsize=(7, 4))
ax.axvline(-2.5, color="gray", linestyle="--", label="viewer colormap clamp")
ax.axvline(2.5, color="gray", linestyle="--")
ax.set_xlabel("regional BAG")
ax.set_title(f"Per-region BAG distribution - {demo_row['uid']}/{demo_row['session_id']}")
ax.legend();

Per-region model quality (`region_metrics.parquet`, stage-1 base-learner OOF diagnostics from the full-data fit).

In [ ]:
region_metrics.sort_values("r2", ascending=False).head(10)

## Step 4 — Static NiiVue demo data

`app/viewer/index.html` is a zero-build page (NiiVue via ESM CDN) that colors the atlas by
`demo_participant.json`, keyed by region name, with a **fixed** diverging colormap
(vmin=-2.5, vmax=2.5, matching `src/neuroalign/visualization/brain.py` - not rescaled per
participant). Inspect the exact data it renders below, then open the page itself to check
visually.

In [ ]:
demo_participant = json.loads((ROOT / "app/assets/demo_participant.json").read_text())
print("uid / session_id:", demo_participant["uid"], demo_participant["session_id"])

regional_bag_vals = demo_participant["regional_bag"]
max_region = max(regional_bag_vals, key=lambda k: abs(regional_bag_vals[k]))
print("most extreme region (regional_bag):", max_region, "=", regional_bag_vals[max_region])
print("  plain name:", reference[max_region]["plain_name"])
print("  centroid MNI:", reference[max_region]["centroid_mni"])

### To view manually

```bash
cd app && python -m http.server 8765   # must serve from app/, not app/viewer/,
                                        # so ../assets/... resolves
```

Then open **http://localhost:8765/viewer/** in a real (GPU-accelerated) browser.

Note: headless verification in this environment (software WebGL / SwiftShader) showed a
rendering artifact - unlabeled/background voxels painted opaque instead of transparent. The
underlying data pipeline was independently verified correct (LUT arrays checked byte-for-byte
in Node against `region_reference.json` + `demo_participant.json` - background alpha=0, all
432 regions present, saturated-region counts matched this notebook's Step 3 numbers exactly).
The artifact did not reproduce reasoning that implicates the code; recommend confirming once
in a real browser.